# Ch 16. 평가셋 만들기

**[원본 웹페이지](https://desty.github.io/study-ai-assistant-engineering/part4/16-evalset/)**

## Abstract

- **Gold set**(정답 포함 평가셋) 을 **30~100건** 규모로 실제 만들기
- **대표성 있는 샘플링** — 난이도 × 도메인 coverage
- **Synthetic vs Human** 레이블링 — 언제 LLM 을 믿고 언제 사람이 필요한가
- 평가셋 **운영** — 버전 관리 · hold-out · 실사용 실패로 갱신
- 평가셋 누출 · 편향 · "너무 쉬운 문제만" 3대 함정

## Dependencies

In [ ]:
!pip install -q anthropic

## API Keys

In [ ]:
import os
from getpass import getpass

if not os.getenv('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY: ')

## 1. 개념 — 평가는 데이터가 절반이다

Ch 15 에서 정한 지표(Recall@5 · Faithfulness · Helpfulness …)는 계산식일 뿐입니다. 실제 숫자가 나오려면 **질문 + 정답** 페어가 필요합니다.

```
gold set = [(질문, 정답 문서 id, 정답 답변, 메타)] × N건
```

평가의 품질은 **이 데이터의 품질**을 절대 넘을 수 없습니다.

## 2. 왜 필요한가 — 수동 테스트의 3가지 한계

**① 확장 불가.** 수동으로 질문 10개 돌려보고 "잘 된다"는 판단은 의미가 없습니다.

**② 재현 불가.** "지난주엔 이 질문에 답 잘하던데" 는 기억에 남은 편향.

**③ 편향된 테스트.** 개발자가 떠올리는 질문은 이미 잘 풀 질문들입니다.

## 3. 어디서 데이터를 가져오나 — 4가지 소스

### 3-1. 실사용 로그 (최우선)
이미 운영 중이면 **실트래픽 질문**이 최고의 소스. 분포가 실제 그대로이기 때문.

### 3-2. 기존 문서에서 역생성
문서를 LLM 에 주고 "이 문서로 답할 수 있는 질문 3개 만들어줘" — synthetic.

### 3-3. 도메인 전문가 브레인스토밍
현업·CS 팀이 "이런 거 자주 물어본다" 를 받는다.

### 3-4. 공개 벤치마크 (보조)
MMLU · TriviaQA 등. **warm-up** 용도만.

**권장 믹스**: 실사용 로그 60% + 전문가 브레인스토밍 30% + 합성/공개 10%

## 4. 최소 예제 — QA 10건으로 시작

In [ ]:
import json
from pathlib import Path

# (1)! 최초 10건 — 손으로 쓴다. 질문 + 정답 문서 id + 모범 답변
SEED = [
    {
        'id': 'q001',
        'question': '환불 정책은 어떻게 되나요?',
        'gold_doc_ids': ['doc-refund-01'],
        'gold_answer': '구매 후 7일 이내 미사용 상품에 한해 환불 가능. 자세한 절차는 마이페이지 > 환불 신청.',
        'difficulty': 'easy',
        'domain': 'policy',
    },
    {
        'id': 'q002',
        'question': '배송비는 얼마인가요?',
        'gold_doc_ids': ['doc-shipping-01'],
        'gold_answer': '5만원 이상 구매 시 무료배송, 미만 시 배송비 2,500원.',
        'difficulty': 'easy',
        'domain': 'policy',
    },
    # ... 더 많은 예제는 실제로 추가
]

def save_evalset(items, path='evalset/qa_v1.jsonl'):  # (2)!
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f'Saved {len(items)} items to {path}')

save_evalset(SEED)

1. **최소 필드 5개** — id · question · gold_doc_ids (검색 평가용) · gold_answer (생성 평가용) · difficulty/domain (샘플링 · 분석용).
2. **JSONL 포맷** — 한 줄 한 건. git diff 에서 변경이 명확하고, stream 처리 가능.

## 5. 실전 튜토리얼 — 100건 gold set 운영

### 5-1. Coverage matrix 먼저 설계
100건을 "되는대로" 모으면 특정 영역에 편중됩니다. **어떤 샘플이 얼마나** 필요한지 **표로 먼저** 정합니다.

난이도 × 도메인 매트릭스에 **목표 건수**를 미리 채워둡니다.

### 5-2. 샘플링 — stratified

In [ ]:
import random
from collections import defaultdict

def stratified_sample(logs, matrix):
    """logs = [{'question', 'difficulty', 'domain'}, ...]
       matrix = {('easy','faq'): 10, ('hard','numeric'): 16, ...}
    """
    buckets = defaultdict(list)
    for log in logs:
        key = (log['difficulty'], log['domain'])
        buckets[key].append(log)

    sampled = []
    for key, target in matrix.items():
        pool = buckets.get(key, [])
        if len(pool) < target:
            print(f'[WARNING] {key}: pool insufficient ({len(pool)}/{target}) — need synthetic')
        sampled.extend(random.sample(pool, min(target, len(pool))))
    return sampled

# 예제
matrix = {
    ('easy', 'faq'): 10,
    ('easy', 'numeric'): 8,
    ('medium', 'faq'): 15,
    ('medium', 'numeric'): 12,
    ('hard', 'faq'): 20,
    ('hard', 'numeric'): 25,
}

print(f'Total target: {sum(matrix.values())} items')

부족한 셀은 합성(§3-2)이나 브레인스토밍으로 **메꿉니다**.

### 5-3. 레이블링 — Human × LLM 하이브리드

**LLM draft → Human review** 가 비용 대비 품질이 좋습니다.

| 단계 | 누가 | 무엇을 |
|---|---|---|
| 1. Draft | LLM (Claude Haiku) | 문서에서 gold_answer 후보 생성 |
| 2. Review | 도메인 담당자 | 사실성·완전성 확인·수정 |
| 3. 2차 검수 | 다른 사람 | 10~20% 샘플 이중 체크 |
| 4. 승인 | PM / Tech Lead | 최종 merge |

### 5-4. Hold-out 분리
평가셋의 **20~30%** 는 **숨겨 둡니다** (hold-out).

- 일반 evalset: 프롬프트 수정 시마다 매번 돌림 → 과적합됨
- Hold-out: 분기에 한 번만 돌려 **진짜 품질** 확인

### 5-5. 버전 관리

In [ ]:
# 버전 관리 구조
version_structure = """
evalset/
  qa_v1.jsonl          # 초안 10건
  qa_v2.jsonl          # 30건 확장
  qa_v3.jsonl          # 100건 · coverage matrix 충족
  qa_holdout.jsonl     # hold-out 30건 (별도)
  CHANGELOG.md         # 무엇을 왜 바꿨는지
  labeling_guide.md
"""
print(version_structure)

git 로 충분합니다 (jsonl 은 diff-friendly). 대용량이면 DVC.

## 6. 자주 깨지는 포인트

### 6-1. 평가셋 누출 (Test set leak)
가장 흔하고 가장 치명적. 증상: "프롬프트 수정할수록 점수가 오르는데 사용자 피드백은 정체."

### 6-2. 편향된 gold
"이 답이 맞다" 의 기준이 레이블러마다 다르면, gold 자체가 흔들립니다.

### 6-3. "쉬운 문제만" 의 함정
개발자는 실패하는 질문을 **피하려는** 본능이 있습니다.

### 6-4. 한 번 만들고 멈추기
"작년에 만든 evalset" 은 유물입니다. **분기마다** 실로그에서 신규 샘플 추가.

## 7. 운영 시 체크할 점

- [ ] 최소 30건 (QA) · 30건 (요약) · 100건 (분류) 의 초기 gold set 이 있는가
- [ ] **Coverage matrix** 로 난이도·도메인 balance 가 시각화됐는가
- [ ] **레이블링 가이드 문서** (1~2페이지) 가 있고 팀에 공유됐는가
- [ ] **Hold-out 20~30%** 가 별도 파일로 분리돼 CI 에서 **안 돌아가는가**
- [ ] **분기 1회** 실로그에서 신규 샘플 수집·레이블링 사이클이 정착됐는가

## 8. 연습문제

### 확인 문제

1. 당신의 도메인(쇼핑·의료·인사 등)을 정하고, Coverage matrix 3×4 를 채워보세요. 각 셀에 목표 건수를 쓰세요.
2. 평가셋이 누출됐다는 **의심 신호** 3가지를 드세요.
3. LLM draft → Human review 하이브리드 레이블링이, 순수 human 보다 왜 **더 나은** 경우가 있는지 설명하세요.
4. 평가셋에 "어려움 30% 이상" 을 강제하는 것이 왜 중요한가요?

---

**다음 챕터** → [Ch 17. LLM-as-a-Judge](../part4/ch17_llm_as_judge.ipynb)